This notebook implements an automated preprocessing pipeline for the Telco Customer Churn dataset using Scikit-learn's Pipeline and ColumnTransformer utilities. The objective is to handle missing values, scale numerical features, and encode categorical variables in a way that is reproducible and free from data leakage, mirroring how preprocessing is structured in production machine learning systems. A custom transformer is also included to resolve a dataset specific formatting issue, demonstrating how Scikit-learn's pipeline framework can be extended beyond its built in tools. The resulting pipeline is designed to be reused directly for model training in later stages of this project

When working with real world datasets, the raw data almost never arrives in a state that a machine learning model can use directly. The Telco Customer Churn dataset is a good example of this. It mixes customer demographics with billing information and service subscriptions, and a few of these fields need cleaning before anything downstream will work correctly. This notebook starts by pulling the dataset directly from a public source so the entire pipeline stays reproducible without needing a manual download step.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Before deciding how to treat each column, it helps to actually look at what is in the data. Running a quick summary reveals the shape of the dataset, which columns hold missing values, and where the data types do not match what they should be. This step often gets skipped by beginners, but skipping it is usually where subtle bugs creep in later, especially when a column that looks numeric is secretly stored as text.

In [ ]:
df.info()
df.isnull().sum()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
count,7043,7043,7043.000000,7043,7043,7043.000000,7043,7043,7043,7043,...,7043,7043,7043,7043,7043,7043,7043,7043.000000,7043,7043
unique,7043,2,NaN,2,2,NaN,2,3,3,3,...,3,3,3,3,3,2,4,NaN,6531,2
top,3186-AJIEK,Male,NaN,No,No,NaN,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,NaN,,No
freq,1,3555,NaN,3641,4933,NaN,6361,3390,3096,3498,...,3095,3473,2810,2785,3875,4171,2365,NaN,11,5174
mean,NaN,NaN,0.162147,NaN,NaN,32.371149,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.761692,NaN,NaN
std,NaN,NaN,0.368612,NaN,NaN,24.559481,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.090047,NaN,NaN
min,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.250000,NaN,NaN
25%,NaN,NaN,0.000000,NaN,NaN,9.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.500000,NaN,NaN
50%,NaN,NaN,0.000000,NaN,NaN,29.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.350000,NaN,NaN
75%,NaN,NaN,0.000000,NaN,NaN,55.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,89.850000,NaN,NaN


Numerical and categorical columns need to be treated very differently during preprocessing. Numerical features benefit from imputation and scaling, while categorical features need a different kind of imputation followed by encoding. Splitting the columns into these two groups up front makes it possible to route each group through its own dedicated pipeline rather than writing repetitive logic for individual columns.

In [ ]:
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_cols = [col for col in df.columns if col not in numerical_cols + ['customerID', 'Churn']]

With the feature groups defined, the next step is to build separate preprocessing pipelines for each type and combine them using a ColumnTransformer. This keeps every transformation step contained and reproducible. Rather than manually imputing and scaling columns one at a time, wrapping this logic inside a Pipeline object means the exact same steps can be applied consistently to new data later, without the risk of accidentally leaking information from the test set into training.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

One issue specific to this dataset is that the TotalCharges column is stored as a string rather than a number, and a handful of entries are blank. Scikit-learn does not have a built in transformer for this kind of dataset specific fix, so writing a small custom transformer class is the cleanest way to handle it. This also demonstrates how Scikit-learn pipelines can be extended beyond the built in tools to handle real world quirks in a dataset.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class TotalChargesFixer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')
        return X

With the custom fix and the ColumnTransformer both defined, they can now be chained together into a single end to end pipeline. Running the full pipeline on the dataset confirms that every transformation step works correctly in sequence, from fixing the malformed column through to the final scaled and encoded output. This is also the version of the pipeline that would later be reused for model training in Task 4, since the preprocessing logic does not need to change once it is wrapped this way.

In [ ]:
full_pipeline = Pipeline(steps=[
    ('fix_charges', TotalChargesFixer()),
    ('preprocessing', preprocessor)
])

X = df.drop(columns=['customerID', 'Churn'])
X_transformed = full_pipeline.fit_transform(X)
print(X_transformed.shape)

(7043, 46)


This pipeline demonstrates a production style approach to preprocessing rather than a one off script. By separating the transformation logic into reusable components, the same pipeline object can be applied to training data, validation data, or entirely new incoming data without rewriting any code. This is the kind of structure that becomes especially valuable once the dataset grows larger or the project moves toward deployment.

And with this our Task#1 of Barakah TechLabs Comes to an end.